In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

%load_ext rpy2.ipython
import rpy2.robjects as ro

def install_and_load_smooth():
    """Install and load the smooth package from CRAN if not already installed"""
    ro.r('''
    if (!requireNamespace("smooth", quietly=TRUE)) {
        install.packages("smooth", repos="https://cran.rstudio.com/")
    }
    library(smooth)
    ''')
    print("Smooth package loaded from CRAN")

install_and_load_smooth()

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython
Smooth package loaded from CRAN


In [22]:
%%R
library(Mcomp)
m1_monthly <- subset(M1, "MONTHLY")

In [23]:
from smooth.adam_general.core.adam import ADAM

DEFAULT_DATA_DIR = "/home/filtheo/smooth/python/tests/speed_tests/benchmark_data"

def load_and_datetime_index_series(dataset_path, series, h,
                                   train_start_date='2023-01-01'):
    train_path = dataset_path + "/" + series + "_train.csv"
    test_path = dataset_path + "/" + series + "_test.csv"

    train_df = pd.read_csv(train_path)
    total_length = len(train_df)
    dates = pd.date_range(start=train_start_date, periods=total_length, freq='M')
    train_df['t'] = dates
    end_date = train_df['t'].iloc[-1]
    train_df.set_index('t', inplace=True)

    test_df = pd.read_csv(test_path)
    dates = pd.date_range(start=end_date + pd.DateOffset(months=1), periods=h, freq='M')
    test_df['t'] = dates
    test_df.set_index('t', inplace=True)
    return train_df, test_df

dataset = 'm1_monthly'
dataset_path = DEFAULT_DATA_DIR + "/" + dataset
metadata = pd.read_csv(dataset_path + "/metadata.csv")
series_ids = metadata['series_id'].tolist()
h = metadata['horizon'].unique()[0]
freq = metadata['frequency'].unique()[0]
print(f"Dataset: {dataset}, series: {len(series_ids)}, h: {h}, freq: {freq}")

Dataset: m1_monthly, series: 617, h: 18, freq: 12


## 1. Single series ARIMA smoke tests

Test several ARIMA specifications on one series to verify Python matches R.

In [24]:
# Pick a series with enough observations
series = series_ids[2]  # series_0003, 129 obs
train_df, test_df = load_and_datetime_index_series(dataset_path, series, h)
print(f"Series: {series}, n={len(train_df)}, h={h}")

# Save to CSV for R
#train_df.to_csv('/tmp/arima_bench_train.csv', index=False)
#test_df.to_csv('/tmp/arima_bench_test.csv', index=False)

Series: series_0003, n=129, h=18


In [25]:
%%R -i train_df
#y_train <- read.csv('/tmp/arima_bench_train.csv')$y
y_train = train_df
arima_specs <- list(
    list(name="ARIMA(1,0,0)",  ar=1, i=0, ma=0),
    list(name="ARIMA(0,0,1)",  ar=0, i=0, ma=1),
    list(name="ARIMA(1,1,0)",  ar=1, i=1, ma=0),
    list(name="ARIMA(0,1,1)",  ar=0, i=1, ma=1),
    list(name="ARIMA(1,1,1)",  ar=1, i=1, ma=1),
    list(name="ARIMA(2,1,0)",  ar=2, i=1, ma=0),
    list(name="ARIMA(0,1,2)",  ar=0, i=1, ma=2),
    list(name="ARIMA(2,1,2)",  ar=2, i=1, ma=2)
)

r_results <- data.frame(
    spec = character(),
    loglik = numeric(),
    coef_str = character(),
    fc_1 = numeric(),
    fc_5 = numeric(),
    stringsAsFactors = FALSE
)

for (spec in arima_specs) {
    tryCatch({
        m <- adam(y_train, model='NNN',
                 orders=list(ar=spec$ar, i=spec$i, ma=spec$ma),
                 lags=1, silent=TRUE)
        fc <- forecast(m, h=18)
        r_results <- rbind(r_results, data.frame(
            spec = spec$name,
            loglik = as.numeric(logLik(m)),
            coef_str = paste(round(coef(m), 6), collapse=", "),
            fc_1 = fc$mean[1],
            fc_5 = fc$mean[5],
            stringsAsFactors = FALSE
        ))
        cat(sprintf("%s: logLik=%.4f, coef=[%s]\n",
                    spec$name, logLik(m), paste(round(coef(m), 6), collapse=", ")))
    }, error = function(e) {
        cat(sprintf("%s: ERROR - %s\n", spec$name, conditionMessage(e)))
    })
}
print(r_results)

ARIMA(1,0,0): logLik=-826.7493, coef=[0.979974]
ARIMA(0,0,1): logLik=-954.0595, coef=[0.827124]
ARIMA(1,1,0): logLik=-821.9784, coef=[-0.292792]
ARIMA(0,1,1): logLik=-807.1509, coef=[-0.820441]


ARIMA(1,1,1): logLik=-804.2686, coef=[0.238155, -0.87554]
ARIMA(2,1,0): logLik=-812.2393, coef=[-0.381081, -0.396807]
ARIMA(0,1,2): logLik=-802.9256, coef=[-0.552884, -0.284306]
ARIMA(2,1,2): logLik=-802.0285, coef=[0.009042, -0.151539, -0.592026, -0.212276]
          spec    loglik                                  coef_str     fc_1
1 ARIMA(1,0,0) -826.7493                                  0.979974 815.3387
2 ARIMA(0,0,1) -954.0595                                  0.827124 609.7792
3 ARIMA(1,1,0) -821.9784                                 -0.292792 694.3876
4 ARIMA(0,1,1) -807.1509                                 -0.820441 734.1281
5 ARIMA(1,1,1) -804.2686                        0.238155, -0.87554 771.2952
6 ARIMA(2,1,0) -812.2393                      -0.381081, -0.396807 803.6788
7 ARIMA(0,1,2) -802.9256                      -0.552884, -0.284306 822.8488
8 ARIMA(2,1,2) -802.0285 0.009042, -0.151539, -0.592026, -0.212276 858.1826
      fc_5
1 751.9637
2   0.0000
3 725.3251
4 734.1281
5 

In [26]:
# Python: same ARIMA specs
arima_specs = [
    ("ARIMA(1,0,0)", 1, 0, 0),
    ("ARIMA(0,0,1)", 0, 0, 1),
    ("ARIMA(1,1,0)", 1, 1, 0),
    ("ARIMA(0,1,1)", 0, 1, 1),
    ("ARIMA(1,1,1)", 1, 1, 1),
    ("ARIMA(2,1,0)", 2, 1, 0),
    ("ARIMA(0,1,2)", 0, 1, 2),
    ("ARIMA(2,1,2)", 2, 1, 2),
]

py_results = []
for name, ar, i_ord, ma in arima_specs:
    try:
        m = ADAM(model='NNN', ar_order=ar, i_order=i_ord, ma_order=ma, lags=[1])
        m.fit(train_df)
        fc = m.predict(h=h)
        coefs = m.adam_estimated['B']
        cf = m.adam_estimated['CF_value']
        py_results.append({
            'spec': name,
            'loglik': cf,
            'coefs': coefs,
            'fc_1': fc['mean'].values[0],
            'fc_5': fc['mean'].values[min(4, len(fc)-1)],
        })
        print(f"{name}: CF={cf:.4f}, coef={np.round(coefs, 6)}")
    except Exception as e:
        print(f"{name}: ERROR - {e}")
        py_results.append({'spec': name, 'loglik': None, 'coefs': None, 'fc_1': None, 'fc_5': None})

df_py = pd.DataFrame(py_results)
df_py

ARIMA(1,0,0): CF=826.7493, coef=[0.979974]
ARIMA(0,0,1): CF=954.0595, coef=[0.827124]
ARIMA(1,1,0): CF=821.9784, coef=[-0.292792]
ARIMA(0,1,1): CF=807.1509, coef=[-0.820441]
ARIMA(1,1,1): CF=804.2686, coef=[ 0.238105 -0.875685]
ARIMA(2,1,0): CF=812.2396, coef=[-0.381323 -0.396645]
ARIMA(0,1,2): CF=802.9256, coef=[-0.552731 -0.284355]
ARIMA(2,1,2): CF=802.0286, coef=[ 0.010982 -0.151898 -0.594058 -0.210457]


,spec,loglik,coefs,fc_1,fc_5
0,"ARIMA(1,0,0)",826.749316,[0.979974365234375],815.338672,751.963746
1,"ARIMA(0,0,1)",954.059453,[0.8271240234375],609.779200,0.000000
2,"ARIMA(1,1,0)",821.978364,[-0.29279238439160893],694.387579,725.325073
3,"ARIMA(0,1,1)",807.150945,[-0.8204413101728589],734.128077,734.128077
4,"ARIMA(1,1,1)",804.268553,"[0.23810492919379, -0.8756845125512511]",771.317768,752.414512
5,"ARIMA(2,1,0)",812.239571,"[-0.38132322225309356, -0.3966445289508208]",803.503007,699.778266
6,"ARIMA(0,1,2)",802.925641,"[-0.5527306578016342, -0.28435489312266804]",822.866618,758.122650
7,"ARIMA(2,1,2)",802.028572,"[0.010981510748463163, -0.15189779700303696, -...",858.073823,756.168295


In [27]:
%%R -o r_results
r_results

          spec    loglik                                  coef_str     fc_1
1 ARIMA(1,0,0)

 -826.7493                                  0.979974 815.3387
2 ARIMA(0,0,1) -954.0595                                  0.827124 609.7792
3 ARIMA(1,1,0) -821.9784                                 -0.292792 694.3876
4 ARIMA(0,1,1) -807.1509                                 -0.820441 734.1281
5 ARIMA(1,1,1) -804.2686                        0.238155, -0.87554 771.2952
6 ARIMA(2,1,0) -812.2393                      -0.381081, -0.396807 803.6788
7 ARIMA(0,1,2) -802.9256                      -0.552884, -0.284306 822.8488
8 ARIMA(2,1,2) -802.0285 0.009042, -0.151539, -0.592026, -0.212276 858.1826
      fc_5
1 751.9637
2   0.0000
3 725.3251
4 734.1281
5 752.3797
6 699.8485
7 758.1339
8 756.1682


In [28]:
# Compare R vs Python
df_compare = df_py[['spec', 'loglik', 'fc_1', 'fc_5']].merge(
    r_results[['spec', 'loglik', 'fc_1', 'fc_5']],
    on='spec', suffixes=('_py', '_r')
)
df_compare['loglik_diff'] = (df_compare['loglik_py'] - df_compare['loglik_r'].abs()).abs()
df_compare['fc1_diff'] = (df_compare['fc_1_py'] - df_compare['fc_1_r']).abs()
df_compare['fc5_diff'] = (df_compare['fc_5_py'] - df_compare['fc_5_r']).abs()

print("ARIMA Single Series Comparison (R vs Python)")
print("=" * 80)
print(df_compare.to_string(index=False))

ARIMA Single Series Comparison (R vs Python)
        spec  loglik_py    fc_1_py    fc_5_py    loglik_r     fc_1_r     fc_5_r  loglik_diff  fc1_diff  fc5_diff
ARIMA(1,0,0) 826.749316 815.338672 751.963746 -826.749316 815.338672 751.963746 0.000000e+00  0.000000  0.000000
ARIMA(0,0,1) 954.059453 609.779200   0.000000 -954.059453 609.779200   0.000000 0.000000e+00  0.000000  0.000000
ARIMA(1,1,0) 821.978364 694.387579 725.325073 -821.978364 694.387579 725.325073 1.136868e-13  0.000000  0.000000
ARIMA(0,1,1) 807.150945 734.128077 734.128077 -807.150945 734.128077 734.128077 1.136868e-13  0.000000  0.000000
ARIMA(1,1,1) 804.268553 771.317768 752.414512 -804.268551 771.295175 752.379684 2.041511e-06  0.022593  0.034828
ARIMA(2,1,0) 812.239571 803.503007 699.778266 -812.239338 803.678751 699.848527 2.330941e-04  0.175745  0.070261
ARIMA(0,1,2) 802.925641 822.866618 758.122650 -802.925641 822.848826 758.133889 3.105924e-07  0.017792  0.011240
ARIMA(2,1,2) 802.028572 858.073823 756.168295 -802.

Initial parameters: -0.269599 -0.269599 



Call:

nloptr(x0 = B, eval_f = CF, lb = lb, ub = ub, opts = list(algorithm = algorithm, 
    xtol_rel = xtol_rel, xtol_abs = xtol_abs, ftol_rel = ftol_rel, 
    ftol_abs = ftol_abs, maxeval = maxevalUsed, maxtime = maxtime, 
    print_level = print_level), etsModel = etsModel, Etype = Etype, 
    Ttype = Ttype, Stype = Stype, modelIsTrendy = modelIsTrendy, 
    modelIsSeasonal = modelIsSeasonal, yInSample = yInSample, 
    ot = ot, otLogical = otLogical, occurrenceModel = occurrenceModel, 
    obsInSample = obsInSample, componentsNumberETS = componentsNumberETS, 
    componentsNumberETSSeasonal = componentsNumberETSSeasonal, 
    componentsNumberETSNonSeasonal = componentsNumberETSNonSeasonal, 
    componentsNumberARIMA = componentsNumberARIMA, lags = lags, 
    lagsModel = lagsModel, lagsModelAll = lagsModelAll, lagsModelMax = lagsModelMax, 
    indexLookupTable = indexLookupTable, profilesRecentTable = profilesRecentTable, 
    matVt = adamCreated$matVt, matWt = adamCreated$matWt, m

## 1b. Diagnostic: Evaluate Python CF at R's optimal coefficients

This confirms whether the cost function itself is identical at the same parameters,
isolating the issue to the optimization convergence path vs a CF computation bug.

In [ ]:
# Parse R coefficients from r_results
r_coefs_by_spec = {}
for _, row in r_results.iterrows():
    coef_str = row['coef_str'].strip()
    coefs = [float(x.strip()) for x in coef_str.split(',')]
    r_coefs_by_spec[row['spec']] = np.array(coefs)

# For each multi-param ARIMA spec, fit Python model, then evaluate CF at R's coefficients
from smooth.adam_general.core.utils.cost_functions import CF
from smooth.adam_general.core.estimator.optimization import _setup_arima_polynomials

diag_results = []
for name, ar, i_ord, ma in arima_specs:
    if ar + ma < 2:
        continue  # Skip single-param models (already match perfectly)

    r_coefs = r_coefs_by_spec[name]

    # Fit Python model to get all internal structures
    m = ADAM(model='NNN', ar_order=ar, i_order=i_ord, ma_order=ma, lags=[1])
    m.fit(train_df)

    # Get Python's optimal B and CF value
    py_B = m.adam_estimated['B'].copy()
    py_CF = m.adam_estimated['CF_value']

    # Build B_r: R's ARIMA coefficients + Python's initial states
    n_arma = ar + ma
    B_r = py_B.copy()
    B_r[:n_arma] = r_coefs[:n_arma]

    # Access internal model structures from ADAM object
    ar_poly, ma_poly = _setup_arima_polynomials(
        m.model_type_dict, m.arima_results, m.lags_dict
    )
    cf_kwargs = dict(
        model_type_dict=m.model_type_dict, components_dict=m.components_dict,
        lags_dict=m.lags_dict, matrices_dict=m.adam_created,
        persistence_checked=m.persistence_results, initials_checked=m.initials_results,
        arima_checked=m.arima_results, explanatory_checked=m.explanatory_dict,
        phi_dict=m.phi_dict, constants_checked=m.constant_dict,
        observations_dict=m.observations_dict, profile_dict=m.profile_dict,
        general=m.general, adam_cpp=m.adam_cpp, bounds=None,
        arPolynomialMatrix=ar_poly, maPolynomialMatrix=ma_poly,
    )

    cf_at_py = CF(B=py_B, **cf_kwargs)
    cf_at_r = CF(B=B_r, **cf_kwargs)
    r_loglik = abs(r_results[r_results['spec'] == name]['loglik'].values[0])

    diag_results.append({
        'spec': name,
        'py_CF_at_py_B': cf_at_py,
        'py_CF_at_r_B': cf_at_r,
        'r_negloglik': r_loglik,
        'CF_diff_at_r_B': abs(cf_at_r - r_loglik),
        'py_B_arma': list(np.round(py_B[:n_arma], 6)),
        'r_B_arma': list(np.round(r_coefs[:n_arma], 6)),
    })

df_diag = pd.DataFrame(diag_results)
print("Diagnostic: Python CF evaluated at R's optimal coefficients")
print("=" * 80)
print("If CF_diff_at_r_B ≈ 0, the cost function is identical and the issue is purely")
print("optimizer convergence path.")
print()
print(df_diag[['spec', 'py_CF_at_py_B', 'py_CF_at_r_B', 'r_negloglik', 'CF_diff_at_r_B']].to_string(index=False))

## 2. ETS+ARIMA hybrid tests

Test hybrid models combining ETS and ARIMA components.

In [ ]:
%%R
hybrid_specs <- list(
    list(name="ANN+AR(1)",    ets="ANN", ar=1, i=0, ma=0),
    list(name="AAN+ARIMA(1,0,1)", ets="AAN", ar=1, i=0, ma=1),
    list(name="ANN+MA(1)",    ets="ANN", ar=0, i=0, ma=1)
)

r_hybrid <- data.frame(
    spec = character(),
    loglik = numeric(),
    coef_str = character(),
    fc_1 = numeric(),
    stringsAsFactors = FALSE
)

for (spec in hybrid_specs) {
    tryCatch({
        m <- adam(y_train, model=spec$ets,
                 orders=list(ar=spec$ar, i=spec$i, ma=spec$ma),
                 lags=1, silent=TRUE)
        fc <- forecast(m, h=18)
        r_hybrid <- rbind(r_hybrid, data.frame(
            spec = spec$name,
            loglik = as.numeric(logLik(m)),
            coef_str = paste(round(coef(m), 6), collapse=", "),
            fc_1 = fc$mean[1],
            stringsAsFactors = FALSE
        ))
        cat(sprintf("%s: logLik=%.4f, coef=[%s]\n",
                    spec$name, logLik(m), paste(round(coef(m), 6), collapse=", ")))
    }, error = function(e) {
        cat(sprintf("%s: ERROR - %s\n", spec$name, conditionMessage(e)))
    })
}

In [ ]:
hybrid_specs_py = [
    ("ANN+AR(1)",         "ANN", 1, 0, 0),
    ("AAN+ARIMA(1,0,1)",  "AAN", 1, 0, 1),
    ("ANN+MA(1)",         "ANN", 0, 0, 1),
]

py_hybrid = []
for name, ets, ar, i_ord, ma in hybrid_specs_py:
    try:
        m = ADAM(model=ets, ar_order=ar, i_order=i_ord, ma_order=ma, lags=[1])
        m.fit(train_df)
        fc = m.predict(h=h)
        coefs = m.adam_estimated['B']
        cf = m.adam_estimated['CF_value']
        py_hybrid.append({'spec': name, 'loglik': cf, 'fc_1': fc['mean'].values[0]})
        print(f"{name}: CF={cf:.4f}, coef={np.round(coefs, 6)}")
    except Exception as e:
        print(f"{name}: ERROR - {e}")
        py_hybrid.append({'spec': name, 'loglik': None, 'fc_1': None})

df_hybrid_py = pd.DataFrame(py_hybrid)
df_hybrid_py

In [ ]:
%%R -o r_hybrid
r_hybrid

In [ ]:
# Compare hybrid models
df_hybrid_compare = df_hybrid_py[['spec', 'loglik', 'fc_1']].merge(
    r_hybrid[['spec', 'loglik', 'fc_1']],
    on='spec', suffixes=('_py', '_r')
)
df_hybrid_compare['loglik_diff'] = (df_hybrid_compare['loglik_py'] - df_hybrid_compare['loglik_r'].abs()).abs()
df_hybrid_compare['fc1_diff'] = (df_hybrid_compare['fc_1_py'] - df_hybrid_compare['fc_1_r']).abs()

print("ETS+ARIMA Hybrid Comparison (R vs Python)")
print("=" * 80)
print(df_hybrid_compare.to_string(index=False))

## 3. Batch ARIMA test across all M1 monthly series

Run ARIMA(1,1,1) on all series and compare R vs Python.

In [ ]:
%%R
# R: ARIMA(1,1,1) on all M1 monthly series
r_arima_batch <- data.frame(
    series_id = integer(),
    loglik = numeric(),
    ar1 = numeric(),
    ma1 = numeric(),
    fc_1 = numeric(),
    stringsAsFactors = FALSE
)

cat(sprintf("Running R ARIMA(1,1,1) on %d series...\n", length(m1_monthly)))

for (sid in 1:length(m1_monthly)) {
    tryCatch({
        temp_series <- m1_monthly[[sid]]
        m <- adam(temp_series, model='NNN',
                 orders=list(ar=1, i=1, ma=1),
                 lags=1, silent=TRUE)
        fc <- forecast(m, h=18)
        cc <- coef(m)
        r_arima_batch <- rbind(r_arima_batch, data.frame(
            series_id = sid,
            loglik = as.numeric(logLik(m)),
            ar1 = cc[1],
            ma1 = cc[2],
            fc_1 = fc$mean[1],
            stringsAsFactors = FALSE
        ))
    }, error = function(e) {
        r_arima_batch <<- rbind(r_arima_batch, data.frame(
            series_id = sid, loglik = NA, ar1 = NA, ma1 = NA, fc_1 = NA,
            stringsAsFactors = FALSE
        ))
    })
    if (sid %% 50 == 0) cat(sprintf("  Processed %d/%d\n", sid, length(m1_monthly)))
}
cat(sprintf("Done. %d series processed.\n", nrow(r_arima_batch)))

In [ ]:
# Python: ARIMA(1,1,1) on all M1 monthly series
py_arima_batch = []

print(f"Running Python ARIMA(1,1,1) on {len(series_ids)} series...")

for idx, series in enumerate(series_ids):
    try:
        train_df_i, test_df_i = load_and_datetime_index_series(dataset_path, series, h)
        m = ADAM(model='NNN', ar_order=1, i_order=1, ma_order=1, lags=[1])
        m.fit(train_df_i)
        fc = m.predict(h=h)
        B = m.adam_estimated['B']
        py_arima_batch.append({
            'series_id': idx + 1,
            'loglik': m.adam_estimated['CF_value'],
            'ar1': B[0] if len(B) > 0 else None,
            'ma1': B[1] if len(B) > 1 else None,
            'fc_1': fc['mean'].values[0],
        })
    except Exception as e:
        py_arima_batch.append({
            'series_id': idx + 1, 'loglik': None,
            'ar1': None, 'ma1': None, 'fc_1': None,
        })
    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(series_ids)}")

df_py_batch = pd.DataFrame(py_arima_batch)
print(f"Done. {len(df_py_batch)} series processed.")

In [ ]:
%%R -o r_arima_batch
r_arima_batch

In [ ]:
# Compare batch results
df_batch = df_py_batch.merge(
    r_arima_batch,
    on='series_id', suffixes=('_py', '_r')
)

df_batch['loglik_diff'] = (df_batch['loglik_py'] - df_batch['loglik_r'].abs()).abs()
df_batch['fc1_diff'] = (df_batch['fc_1_py'] - df_batch['fc_1_r']).abs()

print("ARIMA(1,1,1) Batch Comparison Summary")
print("=" * 80)
print(f"Total series:          {len(df_batch)}")

valid = df_batch.dropna(subset=['loglik_py', 'loglik_r'])
print(f"Both succeeded:        {len(valid)}")
print(f"Mean |loglik diff|:    {valid['loglik_diff'].mean():.4f}")
print(f"Median |loglik diff|:  {valid['loglik_diff'].median():.4f}")
print(f"Max |loglik diff|:     {valid['loglik_diff'].max():.4f}")
print(f"Mean |fc1 diff|:       {valid['fc1_diff'].mean():.4f}")
print(f"Max |fc1 diff|:        {valid['fc1_diff'].max():.4f}")

close_loglik = (valid['loglik_diff'] < 1).sum()
print(f"\n|loglik diff| < 1:     {close_loglik}/{len(valid)} ({100*close_loglik/len(valid):.1f}%)")
close_fc = (valid['fc1_diff'] < 1).sum()
print(f"|fc1 diff| < 1:        {close_fc}/{len(valid)} ({100*close_fc/len(valid):.1f}%)")

In [ ]:
# Show worst mismatches
print("\nTop 10 largest loglik differences:")
print(valid.nlargest(10, 'loglik_diff')[[
    'series_id', 'loglik_py', 'loglik_r', 'loglik_diff',
    'ar1_py', 'ar1_r', 'ma1_py', 'ma1_r'
]].to_string(index=False))

In [ ]:
# Scatter plot: Python vs R log-likelihood
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(valid['loglik_r'].abs(), valid['loglik_py'], alpha=0.5, s=15)
lims = [min(valid['loglik_r'].abs().min(), valid['loglik_py'].min()),
        max(valid['loglik_r'].abs().max(), valid['loglik_py'].max())]
axes[0].plot(lims, lims, 'r--', lw=1)
axes[0].set_xlabel('R |log-likelihood|')
axes[0].set_ylabel('Python CF value')
axes[0].set_title('ARIMA(1,1,1): Log-likelihood comparison')

axes[1].scatter(valid['fc_1_r'], valid['fc_1_py'], alpha=0.5, s=15)
lims = [min(valid['fc_1_r'].min(), valid['fc_1_py'].min()),
        max(valid['fc_1_r'].max(), valid['fc_1_py'].max())]
axes[1].plot(lims, lims, 'r--', lw=1)
axes[1].set_xlabel('R forecast h=1')
axes[1].set_ylabel('Python forecast h=1')
axes[1].set_title('ARIMA(1,1,1): 1-step forecast comparison')

plt.tight_layout()
plt.show()